In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf groq

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from groq import Groq
import os

In [ ]:
os.environ["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"] = "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"

client = Groq(api_key=os.getenv("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"))

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

resumes = []

files = [
    "/content/Makeshwaran Resume.pdf",
    "/content/PRAWINKUMAR-S-RESUME.pdf",
    "/content/PREETHI RESUME.pdf",
    "/content/Resume - 1 (1).pdf",
    "/content/Sibu Resume.pdf"
]

for file in files:
    loader = PyPDFLoader(file)
    resumes.extend(loader.load())

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = splitter.split_documents(resumes)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
vector_db = FAISS.from_documents(documents, embeddings)

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k":10})

In [ ]:
job_description = """
We are looking for a Machine Learning Engineer.

Required Skills:
- Python
- Machine Learning
- Deep Learning
- CNN
- NLP
- TensorFlow or PyTorch
- Data Science
"""

In [ ]:
results = retriever.invoke(job_description)

context = "\n".join([doc.page_content for doc in results])

In [ ]:
prompt = f"""
You are an AI recruiter.

Job Description:
{job_description}

Candidate Resume Information:
{context}

Analyze the candidates and provide:

1. Candidate skills
2. Matching skills with job description
3. Missing skills
4. Score out of 10
5. Rank the candidates

Return results clearly.
"""

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

**Candidate Analysis Results**

### Candidate 1: MAKESHWARAN U

1. **Candidate Skills**: 
   - Python
   - Machine Learning
   - Deep Learning
   - Computer Vision
   - TensorFlow
   - PyTorch
   - Scikit-Learn
   - Keras
   - OpenCV
   - Java
   - JavaScript
   - R
   - SQL
   - C
   - Flask
   - Django
   - React
   - HTML/CSS
   - MySQL
   - Firebase
   - MongoDB
   - Analytical Thinking
   - Problem Solving
   - Team Collaboration

2. **Matching Skills with Job Description**:
   - Python
   - Machine Learning
   - Deep Learning
   - TensorFlow or PyTorch
   - Data Science (through AI and Data Science education)

3. **Missing Skills**:
   - Explicit mention of CNN
   - NLP

4. **Score out of 10**: 8

5. **Rank**: 2

### Candidate 2: PREETHI M

1. **Candidate Skills**:
   - End-to-end machine learning pipelines
   - MLOps practices
   - Computer vision solutions
   - Python (implied through machine learning and data science)
   - AI Intern experience

2. **Matching Skills with Job De